# Classificação de "estadia longa" nas primeiras 24 horas

**Pergunta:** com o que existe até 24 horas depois da entrada na UTI, dá para saber se a estadia de um paciente com insuficiência cardíaca vai passar de X dias?

**O que muda em relação à regressão:** aqui o modelo é treinado direto para essa pergunta (`XGBClassifier`). Na seção anterior, a AUC saía das previsões em dias da regressão.

**Regras**

- O CID só define a coorte. Não entra como variável.
- O alvo é `los_dias > corte`. Os cortes são 3,55 dias (o do artigo de Yang et al., 2025) e 7 dias.
- Mesma divisão por paciente do pré-processamento (`random_state=42`, 80/20). O mesmo paciente não aparece no treino e no teste.
- As variáveis são as mesmas 24 colunas das primeiras 24 horas já salvas em `artefatos/`.
- Os hiperparâmetros e o limiar de decisão são escolhidos só com o treino, com validação cruzada por paciente. O teste só mede o resultado final.

**Métricas:** AUC (principal), AUC-PR, sensibilidade, especificidade e acurácia balanceada no limiar escolhido no treino.

Pré-requisito: ter rodado o pré-processamento, que gera `artefatos/X_treino.csv`, `X_teste.csv`, `y_*.csv` e `ids_*.csv`.

## 1. Configuração e dados

In [ ]:
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import shap
from IPython.display import display
from sklearn.ensemble import RandomForestClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import (
    average_precision_score,
    balanced_accuracy_score,
    confusion_matrix,
    roc_auc_score,
    roc_curve,
)
from sklearn.model_selection import GridSearchCV, GroupKFold, cross_val_predict
from sklearn.preprocessing import StandardScaler
from xgboost import XGBClassifier

RANDOM_STATE = 42
RAIZ = Path.cwd()
ARTEFATOS = RAIZ / "artefatos"
SAIDA = RAIZ / "artefatos_classificacao"
SAIDA.mkdir(exist_ok=True)
CORTES_DIAS = [3.55, 7.0]
N_SHAP = 2000

x_treino = pd.read_csv(ARTEFATOS / "X_treino.csv")
x_teste = pd.read_csv(ARTEFATOS / "X_teste.csv").reindex(columns=x_treino.columns)
x_treino_esc = pd.read_csv(ARTEFATOS / "X_treino_escalonado.csv")
x_teste_esc = pd.read_csv(ARTEFATOS / "X_teste_escalonado.csv").reindex(columns=x_treino_esc.columns)
los_treino = pd.read_csv(ARTEFATOS / "y_treino.csv")["los_dias"]
los_teste = pd.read_csv(ARTEFATOS / "y_teste.csv")["los_dias"]
ids_treino = pd.read_csv(ARTEFATOS / "ids_treino.csv")
ids_teste = pd.read_csv(ARTEFATOS / "ids_teste.csv")

assert len(x_treino) == len(los_treino) == len(ids_treino)
assert len(x_teste) == len(los_teste) == len(ids_teste)
assert not set(ids_treino["subject_id"]) & set(ids_teste["subject_id"]), "paciente repetido"
proibidas = [c for c in x_treino.columns if "los" in c.lower() or "icd" in c.lower() or "diag" in c.lower()]
assert not proibidas, f"coluna proibida em X: {proibidas}"

grupos = ids_treino["subject_id"].to_numpy()
print(f"Treino: {len(x_treino):,} estadias, {ids_treino['subject_id'].nunique():,} pacientes")
print(f"Teste:  {len(x_teste):,} estadias, {ids_teste['subject_id'].nunique():,} pacientes")
print(f"Variáveis: {x_treino.shape[1]}")
for corte in CORTES_DIAS:
    print(
        f"Corte {corte} dias: {100 * (los_treino > corte).mean():.1f}% longas no treino, "
        f"{100 * (los_teste > corte).mean():.1f}% no teste"
    )

## 2. Modelos e limiar de decisão

- **Regressão logística** (baseline linear, X escalonado).
- **Random Forest**.
- **XGBoost**, o modelo principal, com busca de hiperparâmetros por AUC em validação cruzada por paciente (`GroupKFold`, 5 partes).

`scale_pos_weight` compensa o desequilíbrio entre longas e curtas. O limiar de decisão é o que maximiza sensibilidade + especificidade − 1 nas previsões fora da amostra do treino. O teste não participa dessa escolha.

In [ ]:
def limiar_youden(y, prob):
    falsos, verdadeiros, limiares = roc_curve(y, prob)
    return float(limiares[np.argmax(verdadeiros - falsos)])


def metricas_no_limiar(y, prob, limiar):
    predito = (prob >= limiar).astype(int)
    tn, fp, fn, tp = confusion_matrix(y, predito).ravel()
    return {
        "AUC": roc_auc_score(y, prob),
        "AUC_PR": average_precision_score(y, prob),
        "sensibilidade": tp / (tp + fn),
        "especificidade": tn / (tn + fp),
        "acuracia_balanceada": balanced_accuracy_score(y, predito),
        "limiar": limiar,
    }


def treinar_corte(corte):
    y_treino = (los_treino > corte).astype(int).to_numpy()
    y_teste = (los_teste > corte).astype(int).to_numpy()
    peso = float((y_treino == 0).sum() / (y_treino == 1).sum())
    cv = GroupKFold(n_splits=5)

    logistica = LogisticRegression(C=1.0, max_iter=2000, class_weight="balanced", random_state=RANDOM_STATE)
    floresta = RandomForestClassifier(
        n_estimators=300, max_depth=10, min_samples_leaf=10, max_features="sqrt",
        class_weight="balanced", n_jobs=1, random_state=RANDOM_STATE,
    )
    busca = GridSearchCV(
        XGBClassifier(
            objective="binary:logistic", eval_metric="auc", subsample=0.8, colsample_bytree=0.8,
            min_child_weight=5, scale_pos_weight=peso, n_jobs=1, random_state=RANDOM_STATE,
        ),
        {"max_depth": [2, 3, 4], "n_estimators": [200, 400], "learning_rate": [0.03, 0.05]},
        cv=cv, scoring="roc_auc", n_jobs=1,
    )
    busca.fit(x_treino, y_treino, groups=grupos)
    xgb = busca.best_estimator_

    modelos = {
        "Regressão logística": (logistica, x_treino_esc, x_teste_esc),
        "Random Forest": (floresta, x_treino, x_teste),
        "XGBoost": (xgb, x_treino, x_teste),
    }
    linhas, curvas, probs_teste = [], {}, {}
    for nome, (modelo, xtr, xte) in modelos.items():
        oof = cross_val_predict(modelo, xtr, y_treino, groups=grupos, cv=cv, method="predict_proba")[:, 1]
        limiar = limiar_youden(y_treino, oof)
        modelo.fit(xtr, y_treino)
        prob = modelo.predict_proba(xte)[:, 1]
        linhas.append({"corte_dias": corte, "modelo": nome, **metricas_no_limiar(y_teste, prob, limiar)})
        falsos, verdadeiros, _ = roc_curve(y_teste, prob)
        curvas[nome] = (falsos, verdadeiros)
        probs_teste[nome] = prob
    return pd.DataFrame(linhas), curvas, probs_teste, xgb, busca.best_params_, y_teste

## 3. Corte de 3,55 dias

In [ ]:
tab_355, curvas_355, prob_355, xgb_355, params_355, y_355 = treinar_corte(3.55)
print("Melhores hiperparâmetros do XGBoost:", params_355)
display(tab_355.round(3))

## 4. Corte de 7 dias

In [ ]:
tab_7, curvas_7, prob_7, xgb_7, params_7, y_7 = treinar_corte(7.0)
print("Melhores hiperparâmetros do XGBoost:", params_7)
display(tab_7.round(3))

resultado = pd.concat([tab_355, tab_7], ignore_index=True)
resultado.to_csv(SAIDA / "metricas_classificacao.csv", index=False, encoding="utf-8-sig")
pd.DataFrame({"los_dias": los_teste, **{f"xgb_prob_{c}": p for c, p in [(3.55, prob_355["XGBoost"]), (7.0, prob_7["XGBoost"])]}}).to_csv(
    SAIDA / "previsoes_xgboost.csv", index=False, encoding="utf-8-sig"
)

## 5. Classificador treinado direto contra a regressão que virou classificador

A coluna `AUC_regressao` usa as previsões em dias da regressão (`artefatos/previsoes_teste.csv`), no mesmo teste. A diferença mostra quanto treinar para a pergunta certa ajudou.

In [ ]:
previsoes_regressao = pd.read_csv(ARTEFATOS / "previsoes_teste.csv")
assert np.allclose(previsoes_regressao["los_dias"], los_teste), "teste diferente"

linhas = []
for corte, tab in [(3.55, tab_355), (7.0, tab_7)]:
    real = (los_teste > corte).astype(int)
    for modelo_reg, modelo_cls in [("XGBoost", "XGBoost"), ("Random Forest", "Random Forest"), ("Ridge", "Regressão logística")]:
        auc_cls = float(tab.loc[tab["modelo"] == modelo_cls, "AUC"].iloc[0])
        auc_reg = float(roc_auc_score(real, previsoes_regressao[modelo_reg]))
        linhas.append({
            "corte_dias": corte,
            "modelo": modelo_cls,
            "AUC_regressao": round(auc_reg, 3),
            "AUC_classificador": round(auc_cls, 3),
            "ganho": round(auc_cls - auc_reg, 3),
        })
comparacao = pd.DataFrame(linhas)
comparacao.to_csv(SAIDA / "comparacao_regressao_classificador.csv", index=False, encoding="utf-8-sig")
display(comparacao)

## 6. Curvas ROC no teste

In [ ]:
fig, eixos = plt.subplots(1, 2, figsize=(12, 5))
for eixo, (corte, curvas, tab) in zip(eixos, [(3.55, curvas_355, tab_355), (7.0, curvas_7, tab_7)]):
    for nome, (falsos, verdadeiros) in curvas.items():
        auc = float(tab.loc[tab["modelo"] == nome, "AUC"].iloc[0])
        eixo.plot(falsos, verdadeiros, label=f"{nome} (AUC {auc:.3f})")
    eixo.plot([0, 1], [0, 1], "--", color="gray", label="Acaso")
    eixo.set_title(f"Estadia > {corte} dias")
    eixo.set_xlabel("1 - especificidade")
    eixo.set_ylabel("Sensibilidade")
    eixo.legend(loc="lower right")
plt.tight_layout()
plt.savefig(SAIDA / "curvas_roc.png", dpi=150)
plt.show()

## 7. Que variáveis pesam mais (SHAP no XGBoost)

A escala do SHAP é o log da razão de chances de "estadia longa". Valores maiores querem dizer mais peso na decisão.

In [ ]:
def importancia_shap(modelo, nome):
    amostra = x_teste.sample(n=min(N_SHAP, len(x_teste)), random_state=RANDOM_STATE)
    valores = np.asarray(shap.TreeExplainer(modelo)(amostra).values)
    tabela = (
        pd.DataFrame({"variavel": amostra.columns, "shap_medio_abs": np.abs(valores).mean(axis=0)})
        .sort_values("shap_medio_abs", ascending=False)
        .reset_index(drop=True)
    )
    tabela.to_csv(SAIDA / f"shap_{nome}.csv", index=False, encoding="utf-8-sig")
    return tabela


shap_355 = importancia_shap(xgb_355, "corte_3_55")
shap_7 = importancia_shap(xgb_7, "corte_7")

fig, eixos = plt.subplots(1, 2, figsize=(13, 5))
for eixo, (titulo, tabela) in zip(eixos, [("Estadia > 3,55 dias", shap_355), ("Estadia > 7 dias", shap_7)]):
    topo = tabela.head(12).iloc[::-1]
    eixo.barh(topo["variavel"], topo["shap_medio_abs"], color="#1f4e79")
    eixo.set_title(titulo)
    eixo.set_xlabel("SHAP médio absoluto")
plt.tight_layout()
plt.savefig(SAIDA / "shap_classificador.png", dpi=150)
plt.show()

## 8. Como ler

- **AUC 0,5** é o acaso. A AUC diz se o modelo ordena bem os pacientes, sem depender do limiar.
- **Sensibilidade** é a fração das estadias longas que o modelo marca como longas. **Especificidade** é a fração das curtas que ele marca como curtas.
- **Limiar:** foi escolhido no treino. Para outro uso (por exemplo, avisar mais cedo a equipe), o limiar pode mudar. A AUC não muda.
- **Comparação com o artigo de Yang (AUC 0,861):** a coorte, a versão do MIMIC e as variáveis são diferentes. A comparação é de ordem de grandeza, não de igualdade.
- Se a AUC ficar bem abaixo de 0,8, o motivo provável é a informação das primeiras 24 horas. Os dados disponíveis nesse período explicam pouco do tempo total.